# Tutorial 01 (part A): Fully-connected (Dense) neural networks with Tensorflow and Keras
by Dr Ivan Olier

## Introduction
The aim with this tutorial is to get familiar with deep learning model implementations in *Tensorflow/Keras*. With this one in particular, we will practise how to implement dense feedforward neural networks, and what can we do to control for overfitting.

It is good practice to start with a cleaned Python kernel. In order to do it, you can choose one of the options from the *Kernel* menu that start with *Restart kernel ...*.

Let's import the required packages first. We don't need to import everything from the beginning, but it is good practice to do it so.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, InputLayer
from matplotlib import pyplot as plt
import numpy as np
%matplotlib inline

## Dataset
We can learn the basics of *Keras* by walking through a simple example: recognising handwritten digits from the MNIST dataset. MNIST consists of 28 x 28 grayscale images of handwritten digits like these:

![](imgs/mnist_pic.png)

The MNIST dataset is included with *Keras* and can be accessed using the `dataset_mnist()` function.
* Here we load the dataset then create variables for our test and training data:

In [ ]:
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

Let's inspect a few examples. The *MNIST* dataset contains only grayscale images. For more advanced image datasets, we'll have the three color channels (RGB).

In [ ]:
plt.figure(facecolor='w')
for i in range(9):
  plt.subplot(3,3,i+1)
  plt.tight_layout()
  plt.imshow(X_train[i], cmap='gray', interpolation='none')
  plt.title("Digit: {}".format(y_train[i]))
  plt.xticks([])
  plt.yticks([])

In order to train our neural network to classify images we first have to unroll the height $\times$ width pixel format into one big vector - the input vector. So its length must be $28 \cdot 28 = 784$. But let's graph the distribution of our pixel values.

In [ ]:
plt.figure()
plt.hist(X_train[0].reshape(784))
plt.title("Pixel Value Distribution")

Dense feedforward neural networks expect a matrix (2-d array, with rows the observations and columns the features or variables). Therefore, we need to prepare the data for training by converting the 3-d arrays into matrices by reshaping width and height into a single dimension (28x28 images are flattened into length 784 vectors). Then, we convert the greyscale values from integers ranging between 0 to 255 into floating point values ranging between 0 and 1 (that is, we normalise the data):

In [ ]:
X_train = X_train.reshape(60000, 784).astype('float32')
X_test = X_test.reshape(10000, 784).astype('float32')
X_train /= 255
X_test /= 255

Therefore, the new data array dimensions are:

In [ ]:
print("Train matrix shape", X_train.shape)
print("Test matrix shape", X_test.shape)

So far the truth we'll use for training still holds integer values from 0 to 9.

In [ ]:
print(np.unique(y_train, return_counts=True))

Let's encode our categories - digits from 0 to 9 - using *one-hot encoding*. The result is a vector with a length equal to the number of categories. The vector is all zeroes except in the position for the respective category. Thus a '$5$' will be represented by $[0,0,0,0,1,0,0,0,0]$.

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

## Building a first network
We aim to build a first architecture as the one in the below figure:
![](imgs/mlp_mnist1.jpg)

The network consists of:
* An input layer of 784 input features.
* Two hidden layers of 512 neurons each. We will use `relu` activation function for all neurons in the hidden layers.
* An output layer of 10 neurons, one for each class. Their activation functions are `softmax` - they will provide relative class probability.  

In [ ]:
model1 = Sequential()
model1.add(InputLayer(shape=(784,)))
model1.add(Dense(512, activation='relu'))
model1.add(Dense(512, activation='relu'))
model1.add(Dense(10, activation='softmax'))
model1.summary()

* Notice the number of parameters to learn. All of them are either connection weights or biases. Manually verify the number of parameters in the network (with the help of a calculator or a pen and paper).

* We use the `plot_model()` function to visualise the network architecture. Note that it requires the `pydot` and `graphviz` packages to be installed. If you don't have them, you can comment out the line that calls the function.

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model1)

Now, let's set up the optimiser and performance metrics. See the code below:

In [ ]:
model1.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer='adam'
)

We are using *ADAM* this time. We are ready to train the model. Notice in the code below that I'm using `verbose=0`, which means that partial training outcomes won't be prompted. This will significantly increase training speed. I also use that mode here so I do not clutter this document. But feel free to change the verbose mode. You can check the documentation for other options. Other arguments of function `fit` are `epochs` (30, but you could see that a smaller number could also work), `batch_size` (128), and `validation_split` (0.2, 20\% of training subset will be used for validation). The batch size is usually chosen according to the length of the dataset and the expected number of epochs to reach convergence. We have 7,000 rows in the training set hence a batch no larger than 1,000 makes sense. Smaller batches will lead to bumpier learning curves but the algorithm will run faster.

In [ ]:
history = model1.fit(X_train, y_train,
          batch_size=128, epochs=20,
          verbose=1,
          validation_split=0.2)

As before, we can implement a quick function to plot learning curves:

In [ ]:
def plot_history(history):
    plt.figure()
    plt.subplot(2,1,1)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'])
    plt.subplot(2,1,2)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'])
    return;

plot_history(history)

## Exercise

Before we carry on with the rest of the tutorial, let's stop for a minute here to reflect on above results.

1. What are the training and validation losses and accuracies at epoch 10?
2. What can you tell about model performance? Is the model overfitted? Underfitted? comments?
3. Write a line of code to estimate model accuracy on the test subset. How test accuracy compares against training accuracy? comments?

### Solution

In [ ]:
test_scores_model1 = model1.evaluate(X_test, y_test, verbose=0)
print("Test loss model 1:", test_scores_model1[0])
print("Test accuracy model 1:", test_scores_model1[1])
train_scores_model1 = model1.evaluate(X_train, y_train, verbose=0)
print("Train loss model 1:", train_scores_model1[0])
print("Train accuracy model 1:", train_scores_model1[1])

## Controlling for overfitting
Prior model is overfitted. That is, the model has learnt the training set extremely well, but struggles with test sets that have not been seen by the model during learning. As studied in the lecture slides, the key to control for overfitting is to have a good balance between the amount of data available and model complexity (i.e. the number of free parameters that need to be learnt). One way to reduce the risk of overfitting is by getting more data. Depending on the application, this could pose lot of challenges like the increasing cost of recording the data, or perhaps because there is no way to get more (historic data). Alternatively, we can attempt to reduce model complexity. In deep learning, one common way to do it is by dropping neurons out during the training hence the number of free parameters are reduced. We do this in the folowing code:

In [ ]:
from tensorflow.keras.layers import Dropout

model2 = Sequential()
model2.add(InputLayer(shape=(784,)))
model2.add(Dense(512, activation='relu'))
model2.add(Dropout(rate=0.4))
model2.add(Dense(512, activation='relu'))
model2.add(Dropout(rate=0.4))
model2.add(Dense(10, activation='softmax'))

model2.summary()

The function `Dropout` will do exactly that. The dropping rate is controlled by the argument `rate`. Now we can just carry on with the rest of the steps to build the model as before:

In [ ]:
model2.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
history = model2.fit(X_train, y_train,
          batch_size=128, epochs=20,
          verbose=1,
          validation_split=0.2)

If we produce the learning curve plots, we can observe the gap between training and validation has considerably shrank:

In [ ]:
plot_history(history)

Now, we can estimate the performance on the test set:

In [ ]:
test_scores_model2 = model2.evaluate(X_test, y_test, verbose=0)
print("Test loss model 2:", test_scores_model2[0])
print("Test accuracy model 2:", test_scores_model2[1])
train_scores_model2 = model2.evaluate(X_train, y_train, verbose=0)
print("Train loss model 2:", train_scores_model2[0])
print("Train accuracy model 2:", train_scores_model2[1])

## Exercise

* How does the dropout rate choice influence the model performance? Try several rate values in both layers to have some insights. Keep it simple, a couple of extreme values should be enough.

* If you need to clean you *Tensorflow/Keras* session, you can use the following code:


In [ ]:
# reset tensorflow session
from tensorflow.keras import backend as K
K.clear_session()

* Try the original model again. But this time, use a small batch size (e.g. 16). How does it influence the model performance? Why? (note that it will take longer to train the model)
* (Answer below)

In [ ]:
model1 = Sequential()
model1.add(InputLayer(shape=(784,)))
model1.add(Dense(512, activation='relu'))
model1.add(Dense(512, activation='relu'))
model1.add(Dense(10, activation='softmax'))
model1.summary()

model1.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
history = model1.fit(X_train, y_train,
          batch_size=2, epochs=20,
          verbose=1,
          validation_split=0.2)

plot_history(history)

## Exercise 2
* Repeat the above activities, but using the Fashion-MNIST dataset this time.
* Details about the dataset can be found here: https://www.tensorflow.org/datasets/catalog/fashion_mnist
